# Check evidence packets

Change `packet_id`, then restart the kernel and run all cells.
Run the setup cell first; later cells use its variables.
Nothing here changes files or calls a model.

Sources are extracted from PDFs in `private/pdfs/` into `.txt` and `.json` files in `private/sources/`. The JSON records the PDF filename, hashes, extraction details, and page spans. Apple uses `SRC_01`; Microsoft uses `SRC_02`.

In [51]:
import json
from pathlib import Path
from decimal import Decimal
from pprint import pprint

# Find the repo even when Jupyter starts in src/ or a notebook folder.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/benchmark/private/packets").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from inside the agent_test repository.")
folder = repo / "data/benchmark"
packet_id = "EP_0015"  # EP_0001 = Apple, EP_0002 = Microsoft

packet = json.loads((folder / f"private/packets/{packet_id}.json").read_text(encoding="utf-8"))
masked_packet = json.loads((folder / f"public/evidence_packets/{packet_id}.json").read_text(encoding="utf-8"))
source_id = packet["source_id"]
source = (folder / f"private/sources/{source_id}.txt").read_text(encoding="utf-8")
source_info = json.loads((folder / f"private/sources/{source_id}.json").read_text(encoding="utf-8"))

print("Company:", packet["company"]["name"])
print("Original source:", source_info.get("input_file") or source_info.get("source_url"))
print("Saved text:", folder / f"private/sources/{source_id}.txt")

Company: NextEra Energy, Inc.
Original source: private/pdfs/nee-20251231.pdf
Saved text: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/private/sources/SRC_15.txt


## The evidence packet

`private` has the original names and source references. `public` has the masked version the agents will see.

In [52]:
pprint(packet, sort_dicts=False)

{'schema_version': '0.1',
 'packet_id': 'EP_0015',
 'company_alias': 'COMPANY_15',
 'company': {'name': 'NextEra Energy, Inc.', 'ticker': 'NEE'},
 'fiscal_year': 2025,
 'filing': {'form': '10-K',
            'accession_number': '0000753308-26-000015',
            'filing_date': '2026-02-13',
            'source_url': 'https://www.sec.gov/Archives/edgar/data/753308/000075330826000015/nee-20251231.htm'},
 'source_id': 'SRC_15',
 'evidence': [{'evidence_id': 'EP_0015_E01',
               'section': 'Item 7',
               'text': 'OVERVIEW\n'
                       'NEE’s operating performance is driven primarily by the '
                       'operations of its two principal businesses, FPL, which '
                       'serves more than \n'
                       'six million customer accounts in Florida and is the '
                       'largest electric utility in the U.S., and NEER, which '
                       'together with affiliated \n'
                       'entities is

## Source text vs stored evidence

`source_span` is `[start, end]` in the saved source text. The end character is excluded.
Each comparison should print `Equal: True`. Use the printed locator to check the original filing too.

In [53]:
for evidence in packet["evidence"]:
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]

    print("\n---", evidence["evidence_id"], "---")
    print("Location:", evidence["locator"])
    print("Source span:", [start, end])
    print("\nFROM SOURCE:\n", source_excerpt)
    print("\nIN PACKET:\n", evidence["text"])
    print("\nEqual:", source_excerpt == evidence["text"])


--- EP_0015_E01 ---
Location: PDF pages [38]; Overview and GAAP net income and diluted EPS table with fiscal-year, unit and segment headers and interest-allocation footnote
Source span: [175196, 176867]

FROM SOURCE:
 OVERVIEW
NEE’s operating performance is driven primarily by the operations of its two principal businesses, FPL, which serves more than 
six million customer accounts in Florida and is the largest electric utility in the U.S., and NEER, which together with affiliated 
entities is one of the largest energy infrastructure developers in the U.S. The table below presents net income (loss)  attributable 
to NEE and earnings (loss) per s hare attributable to NEE, assuming dilution, by reportable segment, FPL and NEER. Corporate 
and Other is primarily comprised of the operating results of other business activities, as well as other income and expense items, 
including interest expense, and eliminating entries, and may include the net effect of rounding.  See Note 16 for additi

## Source numbers vs stored numbers

Find each recorded source number in the original excerpt and compare it with the stored value.
Also check the printed context: a matching number alone does not prove it belongs to the right row, year, or unit.

In [54]:
for fact in packet["facts"]:
    evidence = next(e for e in packet["evidence"] if e["evidence_id"] == fact["evidence_id"])
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]
    position = source_excerpt.find(fact["source_value"])
    source_number = source_excerpt[position:position + len(fact["source_value"])] if position >= 0 else None

    print("\n---", fact["fact_id"], fact["metric"], "---")
    print("Evidence:", fact["evidence_id"], "|", fact["locator"])
    print("Period:", fact["period"], "| Unit:", fact["unit"])
    if position >= 0:
        print("Source context:\n", source_excerpt[max(0, position - 100):position + len(fact["source_value"]) + 100])
    print("Source number:", source_number)
    print("Packet number:", fact["value"])
    equal = source_number is not None and Decimal(source_number.replace(",", "")) == Decimal(fact["value"])
    print("Equal:", equal)


--- EP_0015_F01 net_income ---
Evidence: EP_0015_E01 | NEE row; Net Income Attributable to NEE columns; FY2025
Period: {'label': 'FY2025', 'start': '2025-01-01', 'end': '2025-12-31'} | Unit: USD_million
Source context:
 299  3,558  1.44  1.12  1.75 
Corporate and Other  (1,152)  104  (800)  (0.56)  0.04  (0.39) 
NEE $ 6,835 $ 6,946 $ 7,310 $ 3.30 $ 3.37 $ 3.60 
______________________
(a) NEER’s results reflect an allocati
Source number: 6,835
Packet number: 6835
Equal: True

--- EP_0015_F02 net_income ---
Evidence: EP_0015_E01 | NEE row; Net Income Attributable to NEE columns; FY2024
Period: {'label': 'FY2024', 'start': '2024-01-01', 'end': '2024-12-31'} | Unit: USD_million
Source context:
 58  1.44  1.12  1.75 
Corporate and Other  (1,152)  104  (800)  (0.56)  0.04  (0.39) 
NEE $ 6,835 $ 6,946 $ 7,310 $ 3.30 $ 3.37 $ 3.60 
______________________
(a) NEER’s results reflect an allocation of in
Source number: 6,946
Packet number: 6946
Equal: True

--- EP_0015_F03 fpl_net_income ---
Evid

## Original vs masked

Read each pair. Names and brands should change; the numbers and meaning should stay the same.

In [55]:
for evidence in packet["evidence"]:
    masked = next(e for e in masked_packet["evidence"] if e["evidence_id"] == evidence["evidence_id"])
    print("\n---", evidence["evidence_id"], "---")
    print("ORIGINAL:\n", evidence["text"])
    print("\nMASKED:\n", masked["text"])

print("\nStored numbers unchanged by masking:")
for fact in packet["facts"]:
    masked_fact = next(f for f in masked_packet["facts"] if f["fact_id"] == fact["fact_id"])
    print(fact["fact_id"], fact["value"], "->", masked_fact["value"], "Equal:", fact["value"] == masked_fact["value"])


--- EP_0015_E01 ---
ORIGINAL:
 OVERVIEW
NEE’s operating performance is driven primarily by the operations of its two principal businesses, FPL, which serves more than 
six million customer accounts in Florida and is the largest electric utility in the U.S., and NEER, which together with affiliated 
entities is one of the largest energy infrastructure developers in the U.S. The table below presents net income (loss)  attributable 
to NEE and earnings (loss) per s hare attributable to NEE, assuming dilution, by reportable segment, FPL and NEER. Corporate 
and Other is primarily comprised of the operating results of other business activities, as well as other income and expense items, 
including interest expense, and eliminating entries, and may include the net effect of rounding.  See Note 16 for additional 
segment information. The following discussion should be read in conjunction with the Notes to Consolidated Financial 
Statements contained herein and all comparisons are with the co

## Notes

Keep any problems found here in the matching report review notebook,
for example `private/reviews/EP_0001_review.ipynb`.
That one notebook holds the evidence checks, report feedback, edits and final output.